In [0]:
# =============================================================================
# jobs/16_build_wiski_lookup.py
#
# Notebook:   16_build_wiski_lookup.py
# Pipeline:   FGS / EA Real-Time Data Pipeline
# Author:     Jon Payne, Environment Agency
# Cadence:    One-off -- re-run only if the source CSV is updated
#
# PURPOSE
# -------
# Joins ea_rt_measure_register with the externally supplied WISKI lookup CSV
# to produce ea_rt_wiski_lookup. This table is the bridge between the EA
# real-time gauge network and systems that identify gauges by WISKI ID rather
# than by station_reference or measure_uri.
#
# WHY A SEPARATE TABLE?
# ---------------------
# The WISKI lookup is maintained externally and covers only a subset of the
# gauge network. Merging it into ea_rt_measure_register would overwrite a
# stable derived table with an external dependency and leave null WISKI
# columns for the majority of gauges. A separate table keeps the register
# clean and makes the WISKI mapping explicit.
#
# JOIN
# ----
# ExternalID (lookup CSV) -> station_reference (ea_rt_measure_register)
# Inner join: only gauges present in both sources enter the output.
# The CSV is the authority on which gauges carry WISKI IDs.
#
# SOURCE
# ------
# config.WISKI_LOOKUP_CSV, by default
# /Volumes/prd_dash_lab/flood_forecasting_unrestricted/
#     shared_external_volume/realTime_WISKI_lookup.csv
#
# OUTPUT TABLE
# ------------
# ea_rt_wiski_lookup
#   measure_uri         -- join key to Bronze and downstream tables
#   station_reference   -- EA RLOI reference
#   station_label       -- human-readable gauge name
#   lat, long           -- WGS84 coordinates
#   external_id         -- ExternalID from the lookup (== station_reference)
#   external_parameter  -- parameter label in the external system
#   parameter           -- parameter type
#   period              -- measurement interval
#   unit                -- unit of measure
#   internal_id         -- WISKI internal ID string
#   internal_parameter  -- parameter label in WISKI
#   import_source       -- source system (e.g. Telemetry)
#   asset_time          -- asset timestamp from the lookup
#   wiski_id            -- WiskiID cast to long; used as join key downstream
#   wiski_parameter     -- parameter code in WISKI
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import config

from pyspark.sql import functions as F

LOOKUP_PATH    = config.WISKI_LOOKUP_CSV
REGISTER_TABLE = config.TBL_EA_RT_REGISTER
OUTPUT_TABLE   = config.TBL_WISKI_LOOKUP

# =============================================================================
# 1. Load source data
# =============================================================================

# Read as all-string initially; we cast explicitly below where type matters.
lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(LOOKUP_PATH)
)

print(f"Lookup rows loaded:   {lookup_df.count()}")
display(lookup_df.limit(5))

# Confirmed columns: measure_uri, station_label, station_reference, lat, long
register_df = spark.table(REGISTER_TABLE)

print(f"Register rows loaded: {register_df.count()}")
display(register_df.limit(5))

In [0]:
# =============================================================================
# 2. Normalise join keys
# =============================================================================

# Trim both sides -- the CSV is externally managed and trailing whitespace
# on ID fields is a common source of silent mismatches.
lookup_df   = lookup_df.withColumn("ExternalID", F.trim(F.col("ExternalID")))
register_df = register_df.withColumn("station_reference", F.trim(F.col("station_reference")))

In [0]:
display(lookup_df)

In [0]:
# =============================================================================
# 3. Join
# =============================================================================

joined_df = register_df.join(
    lookup_df,
    register_df["station_reference"] == lookup_df["ExternalID"],
    how="inner"
)

matched_rows = joined_df.count()
print(f"Matched rows:         {matched_rows}")

# The join is on station only. A station usually has several measures (level,
# flow and so on) and the lookup may have a row per parameter, so a measure can
# be paired with every lookup row for its station. If the two numbers below
# differ, the output has more than one row per measure_uri: check the lookup, and
# consider also matching on parameter, qualifier and period.
matched_measures = joined_df.select(register_df["measure_uri"]).distinct().count()
print(f"Distinct measures:    {matched_measures}")
if matched_rows > matched_measures:
    print(
        f"WARNING: {matched_rows - matched_measures} extra rows. Some measures match "
        f"more than one lookup row, so a measure can carry another parameter's WISKI id."
    )

display(joined_df.limit(10))

In [0]:
# =============================================================================
# 4. Select and cast output columns
# =============================================================================
# WiskiID cast to long -- used as a numeric join key in downstream systems.
# Any value that fails to cast becomes null; the check below flags these rows.
output_df = joined_df.select(
    # From the register
    register_df["measure_uri"],
    register_df["station_reference"],
    register_df["station_label"],
    register_df["lat"],
    register_df["long"],

    # From the lookup -- use lookup_df[] throughout to avoid AMBIGUOUS_REFERENCE
    # where column names collide with those in the register (e.g. Period)
    lookup_df["ExternalID"].alias("external_id"),
    lookup_df["ExternalParameter"].alias("external_parameter"),
    lookup_df["ParameterQualifier"].alias("parameter_qualifier"),
    lookup_df["Period"].alias("period"),
    lookup_df["Unit"].alias("unit"),
    lookup_df["InternalID"].alias("internal_id"),
    lookup_df["InternalParameter"].alias("internal_parameter"),
    lookup_df["ImportSource"].alias("import_source"),
    #lookup_df["AssetTimeSeriesID"].alias("asset_time_series_id"),
    lookup_df["WiskiID"].alias("wiski_id"),
    #lookup_df["WiskiParameter"].alias("wiski_parameter"),
)

display(output_df)

In [0]:
# =============================================================================
# 5. Write
# =============================================================================

# Full overwrite -- this table is derived from a static external CSV.
# Re-run only when the CSV changes; there is no incremental update pattern.
(
    output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print(f"Written {output_df.count()} rows to {OUTPUT_TABLE}")

dbutils.notebook.exit("success")